"""
fine_tune_gpt2_lora.py

Fine-tune GPT-2 Small on Enron email–reply pairs with LoRA adapters,
using either full-thread or subject+last-email prompts and tone conditioning.
"""
need to test the pipeline from local to GitHub

In [ ]:
pip install transformers datasets peft accelerate

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 122.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 105.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 137.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 133.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 133.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 140.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 136.9 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to u

In [ ]:
import argparse
import logging
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType

Data Preparation

In [1]:
#!/usr/bin/env python
# data_prep_step1.py
"""
Step 1: CSV → email/reply pairs (JSONL), robust to column-name variants.

Accepts CSVs with any of these:
- Thread/id column:  message_id | thread_id | email_id
- Text/body column:  clean_message | email_text | text | body
- Tag column:        tag  (values like "<|original|>", "<|reply1|>", ...)

Output JSONL fields per line:
  { "thread": ..., "subject": "", "email": ..., "reply": ..., "tone": "[formal]" }

Defaults:
  input  = data/preprocessed_dataset.csv
  output = data/enron_pairs.jsonl
"""

import argparse
import json
import re
import warnings
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd


# ---------- Resolve repo root (works in .py and in a notebook cell) ----------
def _repo_root() -> Path:
    try:
        return Path(__file__).resolve().parents[1]  # running as a script
    except NameError:
        cwd = Path.cwd()                            # running from a notebook
        return cwd.parent if cwd.name == "src" else cwd

REPO_ROOT = _repo_root()
DEFAULT_INPUT  = REPO_ROOT / "data" / "preprocessed_dataset.csv"
DEFAULT_OUTPUT = REPO_ROOT / "data" / "enron_pairs.jsonl"


# ---------- Helpers ----------
def extract_tag_index(tag: str) -> int:
    """original→0, replyN→N, else→9999 (sorts last)."""
    s = str(tag)
    if s == "<|original|>":
        return 0
    m = re.match(r"<\|reply(\d+)\|>", s)
    return int(m.group(1)) if m else 9999


def _pick_first_present(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


# ---------- Core ----------
def prepare_pairs(input_csv: str | Path, output_jsonl: str | Path, tone: str = "[formal]") -> int:
    """
    Read CSV, group by thread, and write email→reply pairs as JSONL.
    Returns number of examples written.
    """
    input_csv = Path(input_csv)
    output_jsonl = Path(output_jsonl)

    # If relative, interpret from repo root so it works from / or /src
    if not input_csv.is_absolute():
        candidate = (REPO_ROOT / input_csv).resolve()
        if candidate.exists():
            input_csv = candidate

    # Load CSV (local path or URL)
    csv_kwargs = {"engine": "python", "on_bad_lines": "skip", "encoding": "utf-8"}
    read_path = str(input_csv)
    if input_csv.exists():
        pass  # local file found
    else:
        # If it's a URL, pandas can read it directly
        scheme = urlparse(read_path).scheme
        if scheme not in ("http", "https"):
            raise FileNotFoundError(f"Input CSV not found: {input_csv}")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df = pd.read_csv(read_path, **csv_kwargs)

    # Flexible column resolution
    thread_col = _pick_first_present(df, ["message_id", "thread_id", "email_id"])
    text_col   = _pick_first_present(df, ["clean_message", "email_text", "text", "body"])
    tag_col    = "tag" if "tag" in df.columns else None

    if thread_col is None:
        raise KeyError(f"Need a thread id column (message_id/thread_id/email_id). Got: {df.columns.tolist()}")
    if text_col is None:
        raise KeyError(f"Need a text column (clean_message/email_text/text/body). Got: {df.columns.tolist()}")
    if tag_col is None:
        raise KeyError(f"Need a 'tag' column with <|original|>, <|replyN|>. Got: {df.columns.tolist()}")

    # Subject is optional; keep empty for schema stability if missing
    if "subject" not in df.columns:
        df["subject"] = ""

    # Ensure text is string
    df[text_col] = df[text_col].astype(str)

    # Build pairs per thread
    examples = []
    for _, group in df.groupby(thread_col, sort=False):
        g = group.copy()
        g["idx"] = g[tag_col].apply(extract_tag_index)
        g = g.sort_values("idx")

        msgs = g[text_col].tolist()

        # One example per reply position i (i >= 1)
        for i in range(1, len(msgs)):
            examples.append({
                "thread": " ".join(msgs[:i]),  # all prior messages
                "subject": "",                 # not used downstream; kept blank
                "email": msgs[i - 1],          # immediate predecessor
                "reply": msgs[i],              # reply itself
                "tone": tone,                  # default tone tag
            })

    # Write JSONL
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)
    with output_jsonl.open("w", encoding="utf-8") as fout:
        for ex in examples:
            fout.write(json.dumps(ex, ensure_ascii=False) + "\n")

    return len(examples)


# ---------- CLI ----------
def main():
    parser = argparse.ArgumentParser(description="Step 1: Build email→reply pairs from a CSV.")
    parser.add_argument("--input_csv", type=str, default=str(DEFAULT_INPUT),
                        help=f"Path or URL to CSV (default: {DEFAULT_INPUT})")
    parser.add_argument("--output_jsonl", type=str, default=str(DEFAULT_OUTPUT),
                        help=f"Where to write JSONL (default: {DEFAULT_OUTPUT})")
    parser.add_argument("--tone", type=str, default="[formal]",
                        help="Tone token to include (e.g., [formal], [friendly]).")
    # Use parse_known_args so it works inside Jupyter (ignores the -f arg)
    args, _ = parser.parse_known_args()

    input_csv    = Path(args.input_csv)
    output_jsonl = Path(args.output_jsonl)

    print(f"Repo root : {REPO_ROOT}")
    print(f"Reading   : {input_csv}")
    print(f"Writing   : {output_jsonl}")

    n = prepare_pairs(input_csv, output_jsonl, tone=args.tone)
    print(f"Wrote {n} examples → {output_jsonl}")


if __name__ == "__main__":
    main()


Repo root : C:\Jupyter Notebook\enron-email-assist
Reading   : C:\Jupyter Notebook\enron-email-assist\data\preprocessed_dataset.csv
Writing   : C:\Jupyter Notebook\enron-email-assist\data\enron_pairs.jsonl
Wrote 69502 examples → C:\Jupyter Notebook\enron-email-assist\data\enron_pairs.jsonl
